# 8. Edit a query: `replaceClause()` and `removeSubQuery()`

Once a query is built, you don't have to throw it out and rebuild from scratch to change a filter. Two helpers let you transform an existing query into a new one:

- `picsure::replaceClause(query, target, replacement)` swaps every structural match of `target` for `replacement`.
- `picsure::removeSubQuery(query, target)` deletes every structural match of `target`. Empty groups left behind are pruned automatically; an empty result query raises an error.

Both functions are **non-mutating** — they return a new query handle. The original is untouched, so you can fork variants from a common base.

In [1]:
library(picsure)

picsure loaded. On first call, reticulate will provision an isolated Python environment; this takes a few seconds the first time only.



In [2]:
open_hpds_session <- picsure::connect(
  platform = picsure::Platform$BDC_OPEN
)

## Find the two AGE variables we'll swap between

We pick one continuous age variable from each of two studies so we can demonstrate replacing one for the other in a query.

In [3]:
facets <- picsure::facets(open_hpds_session)
picsure::addFacet(facets, "dataset_id", c("phs000810", "phs000007"))

In [4]:
results <- picsure::searchDictionary(open_hpds_session, "age", facets = facets)
results

conceptPath,name,display,description,dataType,studyId,values,min,max,allowFiltering,meta,studyAcronym
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<list>,<dbl>,<dbl>,<lgl>,<chr>,<chr>
\phs000810\pht004715\phv00526256\AGE_IMMI\,phv00526256,AGE_IMMI,Age of immigration among participants who were not born in US mainland (50 US States plus DC),Continuous,phs000810,NULL,0,73,TRUE,NA,HCHSSOL
\phs000007\pht000395\phv00056587\age_s1\,phv00056587,age_s1,Age = StdyDtqa - DOB,Continuous,phs000007,NULL,29,86,TRUE,NA,FHS
\phs000007\pht000397\phv00056723\age_s2\,phv00056723,age_s2,Age (age = formdate - DOB),Continuous,phs000007,NULL,44,86,TRUE,NA,FHS
\phs000007\pht003099\phv00177938\age5\,phv00177938,age5,Age at Exam 5,Continuous,phs000007,NULL,26,96,TRUE,NA,FHS
\phs000007\pht003099\phv00177948\age10\,phv00177948,age10,Age at Exam 10,Continuous,phs000007,NULL,46,102,TRUE,NA,FHS
\phs000007\pht007777\phv00369678\AGE25\,phv00369678,AGE25,Age at Exam 25,Continuous,phs000007,NULL,76,90,TRUE,NA,FHS
\phs000007\pht007777\phv00369677\AGE24\,phv00369677,AGE24,Age at Exam 24,Continuous,phs000007,NULL,74,90,TRUE,NA,FHS
\phs000007\pht007777\phv00369676\AGE23\,phv00369676,AGE23,Age at Exam 23,Continuous,phs000007,NULL,72,90,TRUE,NA,FHS
\phs000007\pht007777\phv00369675\AGE22\,phv00369675,AGE22,Age at Exam 22,Continuous,phs000007,NULL,70,90,TRUE,NA,FHS


In [5]:
# phs000007 -- used in the initial query; later swapped out
age5_phs000007 <- results[results$display == "age5" & results$name == "phv00177938", ]
age5_phs000007_clause <- picsure::buildClause(
  age5_phs000007$conceptPath[[1]],
  type = picsure::PhenotypicFilterType$FILTER,
  min  = 30,
  max  = 40
)

# phs000810 -- the replacement
age_immi_phs000810 <- results[results$display == "AGE_IMMI", ]
age_immi_phs000810_clause <- picsure::buildClause(
  age_immi_phs000810$conceptPath[[1]],
  type = picsure::PhenotypicFilterType$FILTER,
  min  = 30,
  max  = 40
)

## Add a sex variable from FHS

In [6]:
fhs_facet <- picsure::facets(open_hpds_session)
picsure::addFacet(fhs_facet, "dataset_id", "phs000007")

fhs_sex_results <- picsure::searchDictionary(open_hpds_session, "phv00253990", facets = fhs_facet)
fhs_sex_results

conceptPath,name,display,description,dataType,studyId,values,min,max,allowFiltering,meta,studyAcronym
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<list>,<chr>,<chr>,<lgl>,<chr>,<chr>
\phs000007\pht004374\phv00253990\sex\,phv00253990,sex,Sex of the participant,Categorical,phs000007,"Female, Male",NA,NA,TRUE,NA,FHS


## Build the base query: (Female AND age5 30-40) OR (Male AND age5 30-40)

This is the same nested OR-of-ANDs shape from notebook 4.

In [7]:
fhs_sex_male_clause <- picsure::buildClause(
  fhs_sex_results$conceptPath[[1]],
  type       = picsure::PhenotypicFilterType$FILTER,
  categories = list("Male")
)
fhs_male_and_30_to_40 <- picsure::buildClauseGroup(
  list(fhs_sex_male_clause, age5_phs000007_clause),
  operator = picsure::GroupOperator$AND
)

fhs_sex_female_clause <- picsure::buildClause(
  fhs_sex_results$conceptPath[[1]],
  type       = picsure::PhenotypicFilterType$FILTER,
  categories = list("Female")
)
fhs_female_and_30_to_40 <- picsure::buildClauseGroup(
  list(fhs_sex_female_clause, age5_phs000007_clause),
  operator = picsure::GroupOperator$AND
)

fhs_female_or_male_30_to_40 <- picsure::buildClauseGroup(
  list(fhs_female_and_30_to_40, fhs_male_and_30_to_40),
  operator = picsure::GroupOperator$OR
)

fhs_female_or_male_30_to_40$to_query_json()

$operator
[1] "OR"

$phenotypicClauses
$phenotypicClauses[[1]]
$phenotypicClauses[[1]]$operator
[1] "AND"

$phenotypicClauses[[1]]$phenotypicClauses
$phenotypicClauses[[1]]$phenotypicClauses[[1]]
$phenotypicClauses[[1]]$phenotypicClauses[[1]]$phenotypicFilterType
[1] "FILTER"

$phenotypicClauses[[1]]$phenotypicClauses[[1]]$conceptPath
[1] "\\phs000007\\pht004374\\phv00253990\\sex\\"

$phenotypicClauses[[1]]$phenotypicClauses[[1]]$not
[1] FALSE

$phenotypicClauses[[1]]$phenotypicClauses[[1]]$values
[1] "Female"


$phenotypicClauses[[1]]$phenotypicClauses[[2]]
$phenotypicClauses[[1]]$phenotypicClauses[[2]]$phenotypicFilterType
[1] "FILTER"

$phenotypicClauses[[1]]$phenotypicClauses[[2]]$conceptPath
[1] "\\phs000007\\pht003099\\phv00177938\\age5\\"

$phenotypicClauses[[1]]$phenotypicClauses[[2]]$not
[1] FALSE

$phenotypicClauses[[1]]$phenotypicClauses[[2]]$min
[1] 30

$phenotypicClauses[[1]]$phenotypicClauses[[2]]$max
[1] 40



$phenotypicClauses[[1]]$not
[1] FALSE


$phenotypicClauses[[2]]
$phenotypicClauses[[2]]$operator
[1] "AND"

$phenotypicClauses[[2]]$phenotypicClauses
$phenotypicClauses[[2]]$phenotypicClauses[[1]]
$phenotypicClauses[[2]]$phenotypicClauses[[1]]$phenotypicFilterType
[1] "FILTER"

$phenotypicClauses[[2]]$phenotypicClauses[[1]]$conceptPath
[1] "\\phs000007\\pht004374\\phv00253990\\sex\\"

$phenotypicClauses[[2]]$phenotypicClauses[[1]]$not
[1] FALSE

$phenotypicClauses[[2]]$phenotypicClauses[[1]]$values
[1] "Male"


$phenotypicClauses[[2]]$phenotypicClauses[[2]]
$phenotypicClauses[[2]]$phenotypicClauses[[2]]$phenotypicFilterType
[1] "FILTER"

$phenotypicClauses[[2]]$phenotypicClauses[[2]]$conceptPath
[1] "\\phs000007\\pht003099\\phv00177938\\age5\\"

$phenotypicClauses[[2]]$phenotypicClauses[[2]]$not
[1] FALSE

$phenotypicClauses[[2]]$phenotypicClauses[[2]]$min
[1] 30

$phenotypicClauses[[2]]$phenotypicClauses[[2]]$max
[1] 40



$phenotypicClauses[[2]]$not
[1] FALSE



$not
[1] FALSE

In [8]:
picsure::runQuery(open_hpds_session, picsure::buildQuery(phenotypicFilter = fhs_female_or_male_30_to_40))
# Verified with UI: 77 +-3

CountResult(value=78, margin=3, cap=None, raw='78 ±3')

## `replaceClause()` -- swap age5 (phs000007) for AGE_IMMI (phs000810)

Every occurrence of `age5_phs000007_clause` inside the query tree (both AND branches, here) is replaced with `age_immi_phs000810_clause`. The original `fhs_female_or_male_30_to_40` is not modified.

In [9]:
swapped <- picsure::replaceClause(
  fhs_female_or_male_30_to_40,
  age5_phs000007_clause,
  age_immi_phs000810_clause
)
swapped$to_query_json()

$operator
[1] "OR"

$phenotypicClauses
$phenotypicClauses[[1]]
$phenotypicClauses[[1]]$operator
[1] "AND"

$phenotypicClauses[[1]]$phenotypicClauses
$phenotypicClauses[[1]]$phenotypicClauses[[1]]
$phenotypicClauses[[1]]$phenotypicClauses[[1]]$phenotypicFilterType
[1] "FILTER"

$phenotypicClauses[[1]]$phenotypicClauses[[1]]$conceptPath
[1] "\\phs000007\\pht004374\\phv00253990\\sex\\"

$phenotypicClauses[[1]]$phenotypicClauses[[1]]$not
[1] FALSE

$phenotypicClauses[[1]]$phenotypicClauses[[1]]$values
[1] "Female"


$phenotypicClauses[[1]]$phenotypicClauses[[2]]
$phenotypicClauses[[1]]$phenotypicClauses[[2]]$phenotypicFilterType
[1] "FILTER"

$phenotypicClauses[[1]]$phenotypicClauses[[2]]$conceptPath
[1] "\\phs000810\\pht004715\\phv00526256\\AGE_IMMI\\"

$phenotypicClauses[[1]]$phenotypicClauses[[2]]$not
[1] FALSE

$phenotypicClauses[[1]]$phenotypicClauses[[2]]$min
[1] 30

$phenotypicClauses[[1]]$phenotypicClauses[[2]]$max
[1] 40



$phenotypicClauses[[1]]$not
[1] FALSE


$phenotypicClauses[[2]]
$phenotypicClauses[[2]]$operator
[1] "AND"

$phenotypicClauses[[2]]$phenotypicClauses
$phenotypicClauses[[2]]$phenotypicClauses[[1]]
$phenotypicClauses[[2]]$phenotypicClauses[[1]]$phenotypicFilterType
[1] "FILTER"

$phenotypicClauses[[2]]$phenotypicClauses[[1]]$conceptPath
[1] "\\phs000007\\pht004374\\phv00253990\\sex\\"

$phenotypicClauses[[2]]$phenotypicClauses[[1]]$not
[1] FALSE

$phenotypicClauses[[2]]$phenotypicClauses[[1]]$values
[1] "Male"


$phenotypicClauses[[2]]$phenotypicClauses[[2]]
$phenotypicClauses[[2]]$phenotypicClauses[[2]]$phenotypicFilterType
[1] "FILTER"

$phenotypicClauses[[2]]$phenotypicClauses[[2]]$conceptPath
[1] "\\phs000810\\pht004715\\phv00526256\\AGE_IMMI\\"

$phenotypicClauses[[2]]$phenotypicClauses[[2]]$not
[1] FALSE

$phenotypicClauses[[2]]$phenotypicClauses[[2]]$min
[1] 30

$phenotypicClauses[[2]]$phenotypicClauses[[2]]$max
[1] 40



$phenotypicClauses[[2]]$not
[1] FALSE



$not
[1] FALSE

In [10]:
picsure::runQuery(open_hpds_session, picsure::buildQuery(phenotypicFilter = swapped))
# AGE_IMMI is sparser than age5; expect a much smaller cohort
# (typically below the small-cohort obfuscation threshold)

CountResult(value=None, margin=None, cap=10, raw='< 10')

## `removeSubQuery()` -- drop the age filter entirely

Removing the AGE_IMMI clause from `swapped` leaves the two AND groups holding only their sex clauses. The result is effectively `Female OR Male` for FHS participants.

In [11]:
fhs_female_or_male <- picsure::removeSubQuery(swapped, age_immi_phs000810_clause)
fhs_female_or_male$to_query_json()

$operator
[1] "OR"

$phenotypicClauses
$phenotypicClauses[[1]]
$phenotypicClauses[[1]]$operator
[1] "AND"

$phenotypicClauses[[1]]$phenotypicClauses
$phenotypicClauses[[1]]$phenotypicClauses[[1]]
$phenotypicClauses[[1]]$phenotypicClauses[[1]]$phenotypicFilterType
[1] "FILTER"

$phenotypicClauses[[1]]$phenotypicClauses[[1]]$conceptPath
[1] "\\phs000007\\pht004374\\phv00253990\\sex\\"

$phenotypicClauses[[1]]$phenotypicClauses[[1]]$not
[1] FALSE

$phenotypicClauses[[1]]$phenotypicClauses[[1]]$values
[1] "Female"



$phenotypicClauses[[1]]$not
[1] FALSE


$phenotypicClauses[[2]]
$phenotypicClauses[[2]]$operator
[1] "AND"

$phenotypicClauses[[2]]$phenotypicClauses
$phenotypicClauses[[2]]$phenotypicClauses[[1]]
$phenotypicClauses[[2]]$phenotypicClauses[[1]]$phenotypicFilterType
[1] "FILTER"

$phenotypicClauses[[2]]$phenotypicClauses[[1]]$conceptPath
[1] "\\phs000007\\pht004374\\phv00253990\\sex\\"

$phenotypicClauses[[2]]$phenotypicClauses[[1]]$not
[1] FALSE

$phenotypicClauses[[2]]$phenotypicClauses[[1]]$values
[1] "Male"



$phenotypicClauses[[2]]$not
[1] FALSE



$not
[1] FALSE

In [12]:
picsure::runQuery(open_hpds_session, picsure::buildQuery(phenotypicFilter = fhs_female_or_male))
# Verified with UI: ~1267 +-3

CountResult(value=1273, margin=3, cap=None, raw='1273 ±3')